In [6]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("E:/IDEAL")
META_DIR = BASE_DIR / "metadata"
OUT_DIR = BASE_DIR / "processed/aggregated_metadata"
OUT_DIR.mkdir(parents=True, exist_ok=True)

rooms = pd.read_csv(META_DIR / "room.csv")

room_agg = (
    rooms
    .groupby("homeid")
    .agg(
        total_floor_area_raw=("floorarea", "sum"),
        num_rooms=("roomid", "count"),
        avg_room_area_raw=("floorarea", "mean"),
        num_external_windows=("externalwindows", "sum"),
        num_external_walls=("externalwalls", "sum"),
    )
    .reset_index()
)

# 🔧 Κανονικοποίηση εμβαδού (internal IDEAL scale → approx m²)
room_agg["total_floor_area_m2"] = (room_agg["total_floor_area_raw"] / 10).round(1)
room_agg["avg_room_area_m2"] = (room_agg["avg_room_area_raw"] / 10).round(1)

# Κρατάμε ΜΟΝΟ τα κανονικοποιημένα
room_agg = room_agg[
    [
        "homeid",
        "total_floor_area_m2",
        "num_rooms",
        "avg_room_area_m2",
        "num_external_windows",
        "num_external_walls",
    ]
]

room_agg.to_csv(OUT_DIR / "room_aggregated.csv", index=False)

room_agg.head()

,homeid,total_floor_area_m2,num_rooms,avg_room_area_m2,num_external_windows,num_external_walls
0,47,42.7,5,8.5,3,3
1,59,76.0,6,12.7,5,5
2,61,68.5,9,7.6,5,5
3,62,81.5,7,11.6,4,4
4,63,72.5,9,8.1,6,12


In [3]:
appliances = pd.read_csv(META_DIR / "appliance.csv")

appliance_agg = (
    appliances
    .groupby("homeid")
    .agg(
        num_appliances=("number", "sum"),
        num_electric_appliances=("number", lambda x: x[appliances.loc[x.index, "powertype"] == "electric"].sum()),
        num_gas_appliances=("number", lambda x: x[appliances.loc[x.index, "powertype"] == "gas"].sum()),
    )
    .reset_index()
)

appliance_agg.head()

,homeid,num_appliances,num_electric_appliances,num_gas_appliances
0,47,14,6,8
1,55,13,8,5
2,59,10,6,4
3,61,17,8,9
4,62,21,8,13


In [4]:
persons = pd.read_csv(META_DIR / "person.csv")

def age_group(ageband):
    if ageband == "0-4":
        return "child"
    if ageband in ["20-24", "25-29", "30-34", "35-39", "40-44", "45-49", "50-54", "55-59", "60-64"]:
        return "adult"
    if ageband in ["65-69", "70-74", "75-79", "80-84", "85+"]:
        return "elderly"
    return None

persons["age_group"] = persons["ageband"].apply(age_group)

person_agg = (
    persons
    .groupby("homeid")
    .agg(
        num_children=("age_group", lambda x: (x == "child").sum()),
        num_adults=("age_group", lambda x: (x == "adult").sum()),
        num_elderly=("age_group", lambda x: (x == "elderly").sum()),
        num_working_adults=("workingstatus", lambda x: (x == "Paid work").sum()),
        num_full_time_workers=("weeklyhoursofwork", lambda x: x.isin(["41-50"]).sum()),
        num_part_time_workers=("weeklyhoursofwork", lambda x: x.isin(["1-10", "11-20", "21-30"]).sum()),
        num_males=("gender", lambda x: (x == "Male").sum()),
        num_females=("gender", lambda x: (x == "Female").sum()),
    )
    .reset_index()
)

person_agg.head()


,homeid,num_children,num_adults,num_elderly,num_working_adults,num_full_time_workers,num_part_time_workers,num_males,num_females
0,47,0,2,0,2,1,0,1,1
1,59,1,2,0,2,1,0,1,1
2,61,0,2,0,2,0,0,1,1
3,62,0,2,0,2,0,0,1,1
4,63,0,2,0,2,1,0,1,1


In [5]:
OUT_DIR = BASE_DIR / "processed/aggregated_metadata"
OUT_DIR.mkdir(parents=True, exist_ok=True)

room_agg.to_csv(OUT_DIR / "room_aggregated.csv", index=False)


appliance_agg.to_csv(OUT_DIR / "appliance_aggregated.csv", index=False)

person_agg.to_csv(OUT_DIR / "person_aggregated.csv", index=False)

import os
os.listdir(OUT_DIR)

['room_aggregated.csv', 'appliance_aggregated.csv', 'person_aggregated.csv']